In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_tavily import TavilySearch

In [3]:
_tavily = TavilySearch(max_results=2)

@tool
def tavily_search(query: str) -> str:
    """Search the web for current information about a topic."""
    return str(_tavily.invoke({"query": query}))

tool_instance = tavily_search

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_ollama)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_ollama(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            args = {"query": t['args']['query']}
            result = self.tools[t['name']].invoke(args)
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOllama(model="llama3.1:8b")
abot = Agent(model, [tool_instance], system=prompt, checkpointer=memory)

In [8]:
messages = [HumanMessage(content="what is the weather in seoul?")]

In [9]:
thread = {"configurable": {"thread_id": "1"}}

In [10]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T05:27:38.524513Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6424745125, 'load_duration': 5369940791, 'prompt_eval_count': 223, 'prompt_eval_duration': 572754708, 'eval_count': 21, 'eval_duration': 441141124, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaad9-a9ff-72a1-9418-4f238cf9bfd1-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'seoul weather'}, 'id': '1cf9dae1-d8d9-48af-bc23-3f263eda5c52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 21, 'total_tokens': 244})]
Calling: {'name': 'tavily_search', 'args': {'query': 'seoul weather'}, 'id': '1cf9dae1-d8d9-48af-bc23-3f263eda5c52', 'type': 'tool_call'}
Back to the model!
[ToolMessage(content='{\'query\': \'seoul weather\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': 

In [11]:
messages = [HumanMessage(content="what is the weather in jeju?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T05:29:19.570911Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1204718500, 'load_duration': 162304542, 'prompt_eval_count': 1172, 'prompt_eval_duration': 504314416, 'eval_count': 20, 'eval_duration': 472850419, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaadb-491a-7fd0-9ec5-76b136f6a6ea-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'jeju weather'}, 'id': '04644720-3a4e-411c-b60d-e37e0f40bc11', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1172, 'output_tokens': 20, 'total_tokens': 1192})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'jeju weather'}, 'id': '04644720-3a4e-411c-b60d-e37e0f40bc11', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'jeju weather\', \'follow_up_questions\': None, \'answer\': None, \'i

In [12]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T05:29:53.930496Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1035252166, 'load_duration': 94870375, 'prompt_eval_count': 1935, 'prompt_eval_duration': 403838416, 'eval_count': 23, 'eval_duration': 508295043, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaadb-cffa-7702-925b-cfecad84fb9c-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'seoul vs jeju temperature'}, 'id': '8a4f326e-ce33-47de-ad2a-817acce40d92', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1935, 'output_tokens': 23, 'total_tokens': 1958})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'seoul vs jeju temperature'}, 'id': '8a4f326e-ce33-47de-ad2a-817acce40d92', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='{\'query\': \'seoul vs jeju temperature\', \'follow_up_qu

In [13]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-06-09T05:30:26.978377Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1074609458, 'load_duration': 93671125, 'prompt_eval_count': 220, 'prompt_eval_duration': 407413625, 'eval_count': 26, 'eval_duration': 556396461, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019eaadc-50e7-7dd1-b75d-033ff94509b9-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'which city or location has higher temperatures?'}, 'id': 'c1ae4d30-0d48-4518-bda2-6bdc41502662', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 26, 'total_tokens': 246})]}
Calling: {'name': 'tavily_search', 'args': {'query': 'which city or location has higher temperatures?'}, 'id': 'c1ae4d30-0d48-4518-bda2-6bdc41502662', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content="{'query': 'which

Streaming tokens

In [14]:
memory = MemorySaver()
abot = Agent(model, [tool_instance], system=prompt, checkpointer=memory)

In [15]:
messages = [HumanMessage(content="what is the weather in seoul?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            print(content, end="|")

/Users/baejinho/Documents/GitHub/ai-agent-cloud/14주차/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3746: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


Calling: {'name': 'tavily_search', 'args': {'query': 'seoul weather'}, 'id': '1d81a4b4-50cc-4311-bf0c-d29ed6277894', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| Seoul| is| partly| cloudy| with| a| temperature| of| |27|.|3|°C| (|81|.|1|°F|).| The| humidity| is| at| |30|%| and| the| wind| speed| is| |14|.|8| km|/h| (|9|.|2| mph|)| from| the| west|.| There| is| no| precipitation| expected| today|.| 

|However|,| please| note| that| the| weather| forecast| is| subject| to| change| and| it|'s| always| a| good| idea| to| check| the| latest| update| before| planning| your| day|.|